# Классы и ООП: перегрузка операторов и проектирование

## Цели лекции

После этой лекции вы сможете:
- научить свои классы работать с операторами и встроенными функциями:
  `+`, `==`, `<`, `[]`, `in`, `len()`, `print()`, `with`, вызов `obj()`;
- делать объекты сравнимыми, сортируемыми и пригодными как ключи словаря;
- управлять доступом к атрибутам: `__getattr__`, `__setattr__`, дескрипторы;
- выбирать между наследованием, композицией и делегированием;
- описывать классы-данные через `dataclass` и наборы значений через `enum`.

### Основы перегрузки операторов

Что такое специальные методы, знакомо по `__init__` из лекции 7: Python вызывает
их сам в ответ на встроенные операции. Перегрузка операторов – это
определение таких методов в своём классе, чтобы его объекты работали с `+`, `[]`,
`len()`, `print()` так же, как встроенные типы.

- Без своих методов операции вроде `+`, `[]` и `len()` для объекта не работают
  (`TypeError`). У `==`, `repr()`, `str()` и `bool()` есть поведение по умолчанию
  от `object`: равенство – это тождество, а любой объект истинен.
- С перегрузкой код со своими типами такой же короткий, как со встроенными:
  `a + b` вместо `a.add(b)`.

In [1]:
# Перегрузка | – объединение наборов флагов
class Flags:
    def __init__(self, names):
        self.names = set(names)

    def __or__(self, other):
        return Flags(self.names | other.names)

    def __repr__(self):
        return f"Flags({sorted(self.names)})"


print(Flags(["read"]) | Flags(["write"]))

Flags(['read', 'write'])


In [11]:
# __new__ создаёт объект, __init__ заполняет уже созданный. Свой __new__ нужен
# редко: для наследников неизменяемых типов (int, str, tuple) и для Одиночки
# (лекция 7в)
class CustomNew:
    def __new__(cls, *args, **kwargs):
        print("Создание объекта (этап __new__)")
        instance = super().__new__(cls)   # объект создаёт родитель – object; cls – его класс
        return instance

    def __init__(self):
        print("Инициализация объекта (этап __init__)")

obj = CustomNew()

Создание объекта (этап __new__)
Инициализация объекта (этап __init__)


### Сравнение: `__eq__`, `__lt__` и другие

Операторы сравнения вызывают методы `__eq__` (`==`), `__ne__` (`!=`), `__lt__` (`<`),
`__le__` (`<=`), `__gt__` (`>`), `__ge__` (`>=`). Без них `==` сравнивает тождество
объектов, а `<` выдаёт `TypeError`.

`!=` Python выводит из `__eq__` сам. Сортировке достаточно `__lt__`: `sorted`
сравнивает элементы только через `<`.

In [1]:
# Сравнение и сортировка версий по компонентам
class Version:
    def __init__(self, *parts):
        self.parts = parts

    def __eq__(self, other):
        return self.parts == other.parts

    def __lt__(self, other):
        return self.parts < other.parts

    def __repr__(self):
        return "Version" + str(self.parts)


# Версии сравниваются как кортежи чисел: (3, 12) > (3, 9).
# Строки сравнились бы посимвольно, и вышло бы "3.12" < "3.9"
a = Version(3, 12)
b = Version(3, 9)

In [1]:
print(a == b)
print(a < b, b < a)
print(sorted([a, b]))      # sorted опирается на __lt__

False
False True
[Version(3, 9), Version(3, 12)]


### Остальные сравнения: `functools.total_ordering`

Сравнений шесть, а `Version` определяет два. `>` сработает: Python перевернёт
сравнение и вызовет `__lt__` у правого операнда. А `<=` и `>=` так не получить.

Декоратор `@functools.total_ordering` достраивает недостающие сравнения
по `__eq__` и одному из `__lt__`, `__le__`, `__gt__`, `__ge__`.

In [5]:
import functools

try:
    print(a <= b)
except TypeError as error:
    print("TypeError:", error)


@functools.total_ordering
class OrderedVersion(Version):
    pass                       # __eq__ и __lt__ унаследованы, остальное достроено


x, y = OrderedVersion(3, 12), OrderedVersion(3, 9)
print(x <= y, x >= y, x > y)

TypeError: '<=' not supported between instances of 'Version' and 'Version'
False True True


### Равенство не гарантирует хешируемость

Хешируемость встречалась в лекции 2: элементы множества должны быть хешируемыми,
то же требование – к ключам словаря. **Хеш** – целое число, которое `hash(x)`
вычисляет по значению объекта; по нему словарь и множество сразу находят место
элемента. В лекции 2 хешируемость упрощали до неизменяемости. Для своего класса
точнее: объект хешируем, если у класса есть `__hash__`. По умолчанию он есть
и вычисляется из `id` объекта.

Но если класс определил `__eq__` и не определил `__hash__`, Python устанавливает
`__hash__ = None`. Проверим уже созданный `Version`: сравнение работает,
а использование ключом словаря запрещено.

In [1]:
version = Version(3, 12)
same_version = Version(3, 12)
print("Равны:", version == same_version)
print("Хеш отключён:", Version.__hash__ is None)
try:
    releases = {version: "стабильная"}
except TypeError:
    print("Version нельзя использовать как ключ словаря")

Равны: True
Хеш отключён: True
Version нельзя использовать как ключ словаря


### Правила ключа со своим равенством

Для объектов, которые поддерживают `hash`, должны выполняться два условия:

- Если `a == b`, то `hash(a) == hash(b)`. Обратное неверно: разные объекты
  могут иметь одинаковый хеш, поэтому словарь также проверяет равенство.
- Хеш объекта не меняется за время его жизни. Поля, участвующие в равенстве
  и хеше ключа, должны сохранять значения.

Если равенство определяется набором полей, хеш обычно вычисляют от кортежа
этих же полей: например, `hash((self.code, self.region))`.

Просто вернуть `id(self)` нельзя: `id` – номер конкретного объекта, и у двух
разных объектов он разный, даже если они равны. А хеш у равных объектов обязан
совпадать.

`Version` мы `__hash__` не добавляем: поле `parts` можно переприсвоить, и хеш ключа
разошёлся бы с содержимым. Так что `Version` остаётся нехешируемым, как показал
предыдущий экран. Ключ со стабильными полями – в разделе `dataclass`
(«Ключ из неизменяемых хешируемых полей»).

### Контекстные менеджеры: `__enter__` и `__exit__`

`with obj as name:` вызывает `obj.__enter__()` и связывает его результат с `name`.
После блока – даже если в нём возникло исключение – вызывается `obj.__exit__(...)`.
Так `with open(...)` закрывает файл.

Свой класс становится контекстным менеджером, если определяет эти два метода.

`__exit__` получает сведения об исключении (или три `None`), а его результат отвечает
на вопрос «исключение обработано?». `True` – да: оно гасится, и программа идёт дальше
после блока. `False` или `None` – нет: исключение летит дальше. Так устроен
`contextlib.suppress(ТипОшибки)`: он гасит только указанный тип.

In [8]:
class Resource:
    def __enter__(self):
        print("Ресурс открыт")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        # exc_type, exc_value, traceback – сведения об исключении или None
        print("Ресурс закрыт")
        return False  # если вернуть True – исключение будет подавлено


with Resource() as res:          # res – то, что вернул __enter__
    print("Работаем с ресурсом", type(res).__name__)

try:
    with Resource():
        raise ValueError("ошибка внутри with")
except ValueError as error:
    print("Исключение дошло до вызывающего кода:", error)

Ресурс открыт
Работаем с ресурсом Resource
Ресурс закрыт
Ресурс открыт
Ресурс закрыт
Исключение дошло до вызывающего кода: ошибка внутри with


### Контекст-менеджер через генератор: `contextlib.contextmanager`

Писать класс с `__enter__`/`__exit__` ради простого менеджера – многословно.
Декоратор `@contextlib.contextmanager` позволяет описать контекст-менеджер
**одной функцией-генератором**:
- код **до** `yield` выполняется при входе в `with`, как `__enter__`;
- на `yield` функция приостанавливается – в этот момент выполняется тело блока `with`;
- код **после** `yield` (обычно в `finally`) выполняется при выходе, как `__exit__` –
  даже при ошибке.

In [9]:
import contextlib
import time

# finally – как в timer_always из лекции 6: время печатается и при ошибке
@contextlib.contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield                      # пауза: сейчас выполняется тело блока with
    finally:
        print(f"{label}: {time.perf_counter() - start:.4f} сек")

with timer("суммирование"):
    total = sum(range(1_000_000))
print("Результат:", total)

суммирование: 0.0118 сек
Результат: 499999500000


### Индексация и срезы: `__getitem__` и `__setitem__`

- `X[i]` вызывает метод `__getitem__(i)` – класс сам определяет, что вернуть.  
- `X[i:j:k]` (срез) вызывает тот же метод, но с объектом `slice(start, stop, step)`.  
- Синтаксис среза – это просто «сахар» над `X[slice(i, j, k)]`.  
- Внутри метода можно различать:  
  - `int` – доступ по индексу,  
  - `slice` – обработка среза через `index.start`, `index.stop`, `index.step`.  
- `__setitem__` перехватывает операции присваивания по индексу и срезу.  
- Оба метода часто делегируют логику внутренним структурам (например, списку).  


In [12]:
# Индексация, срезы и присваивания

class Indexer:
    def __init__(self, data):
        self.data = list(data)

    def __getitem__(self, index):
        print("getitem ->", index)
        return self.data[index]  # работает и для int, и для slice

    def __setitem__(self, index, value):
        print("setitem ->", index, "=", value)
        self.data[index] = value

    def __repr__(self):
        return f"Indexer({self.data})"


X = Indexer([5, 6, 7, 8, 9])

# Индексация
print(X[0])
print(X[-1])


getitem -> 0
5
getitem -> -1
9


In [13]:
# Срезы
print(X[2:4])
print(X[::2])
print(X[slice(1, None)])

getitem -> slice(2, 4, None)
[7, 8]
getitem -> slice(None, None, 2)
[5, 7, 9]
getitem -> slice(1, None, None)
[6, 7, 8, 9]


In [14]:
# Присваивание по индексу и срезу
X[0] = 100
X[-2:] = [888, 999, 111]
print("После изменений:", X)

setitem -> 0 = 100
setitem -> slice(-2, None, None) = [888, 999, 111]
После изменений: Indexer([100, 6, 7, 888, 999, 111])


In [15]:
# Пример класса, различающего int и slice
class VerboseGet:
    def __getitem__(self, index):
        if isinstance(index, int):
            print("индексация:", index)
            return index ** 2
        print("срез:", index.start, index.stop, index.step)
        return list(range(index.start or 0, index.stop or 0, index.step or 1))

V = VerboseGet()
print(V[3])
print(V[1:5:2])

индексация: 3
9
срез: 1 5 2
[1, 3]


### Метод `__index__`: объект как целое число

`__index__` возвращает целое число, которое Python подставляет вместо объекта там,
где нужен именно `int`: в индексах и срезах (`L[obj]`, `L[obj:]`), в `bin()`, `hex()`,
`oct()`, `range()`. С `__getitem__` он не связан.

С ним свой целочисленный тип можно использовать как индекс списка – так устроены,
например, целые числа NumPy.

In [16]:
# Демонстрация метода __index__
class C:
    def __index__(self):
        print("Вызван __index__()")
        return 255

X = C()
print("hex(X):", hex(X))
data = [f"элемент {i}" for i in range(256)]
print("data[X]:", data[X])

Вызван __index__()
hex(X): 0xff
Вызван __index__()
data[X]: элемент 255


### Итерация через индекс: `__getitem__`

- Если в классе **нет** `__iter__`, цикл `for` вызывает `__getitem__(0)`,
  `__getitem__(1)` и так далее, пока метод не выбросит `IndexError`.
- Поэтому такой объект работает в `for`, во включениях, в `list()`, `tuple()`,
  `map()` и с оператором `in`.
- Условие: `__getitem__` принимает целые индексы с нуля и сообщает конец через
  `IndexError`. Класс без `__iter__` со строковыми ключами так не обойти: `for`
  получит `KeyError` уже на индексе 0.

In [19]:
# Демонстрация итерации через __getitem__

class StepperIndex:
    def __init__(self, data):
        self.data = data

    def __getitem__(self, i):
        print(f"__getitem__({i}) вызван")
        return self.data[i]


X = StepperIndex("hack")

# Индексация
print("X[1] =", X[1])


__getitem__(1) вызван
X[1] = a


In [20]:
# Итерация через for
for ch in X:
    print("получен:", ch)

__getitem__(0) вызван
получен: h
__getitem__(1) вызван
получен: a
__getitem__(2) вызван
получен: c
__getitem__(3) вызван
получен: k
__getitem__(4) вызван


In [21]:
# Другие формы итерации
# list() вызывает __getitem__(0), (1), … пока не получит IndexError:
# вызовов на один больше, чем элементов
print("Список через list(X):", list(X))

__getitem__(0) вызван
__getitem__(1) вызван
__getitem__(2) вызван
__getitem__(3) вызван
__getitem__(4) вызван
Список через list(X): ['h', 'a', 'c', 'k']


### Итерируемые объекты: `__iter__` и `__next__`

Протокол итерации разбирали в лекции 4б: `iter(obj)` вызывает `__iter__` и получает
итератор, `next()` вызывает его `__next__`, конец сообщается через `StopIteration`.
Здесь реализуем протокол сами. Если у класса есть `__iter__`, запасной путь
через `__getitem__` из предыдущего раздела не используется.

In [23]:
# Пример итерируемого объекта

class Squares:
    def __init__(self, start, stop):
        self.value = start - 1   # текущее состояние
        self.stop = stop         # граница итерации

    def __iter__(self):
        return self              # итератор – сам объект: состояние обхода в self.value

    def __next__(self):
        if self.value == self.stop:
            raise StopIteration  # завершение итерации
        self.value += 1
        return self.value ** 2   # возвращаем следующий квадрат


In [24]:
for num in Squares(1, 5):
    print(num, end=" ")

print()

1 4 9 16 25 


In [26]:
# Преобразование в список
print(list(Squares(1, 5)))

# Индексация невозможна:
try:
    print(Squares(1, 5)[1])
except TypeError as e:
    print("Ошибка:", e)

[1, 4, 9, 16, 25]
Ошибка: 'Squares' object is not subscriptable


### Однократная и многократная итерация. Классы и генераторы

- Если `__iter__` возвращает `self`, объект поддерживает **только один проход**:
  после первой итерации он исчерпан, и повторный обход ничего не даст.
  Чтобы начать заново, нужен новый экземпляр.
- Для **многократной** итерации `__iter__` должен при каждом вызове возвращать
  **новый** итератор: отдельный объект-итератор, `iter()` по внутреннему списку
  или генератор. Короче всего – написать `__iter__` с `yield`: каждый вызов
  создаёт новый генератор.

> Деление на однопроходные и многопроходные разбиралось в лекции про итераторы
> на встроенных объектах: `range` выдерживает много проходов, `zip` и `enumerate` –
> один. Здесь то же свойство мы задаём сами, выбирая, что вернёт `__iter__`.

In [27]:
# Класс с однократной итерацией – Squares из предыдущего раздела:
# __iter__ возвращает self, поэтому объект проходится один раз
X = Squares(1, 5)
print(list(X))                  # первая итерация
print(list(X))                  # вторая – объект уже исчерпан
print(list(Squares(1, 5)))      # заново – только с новым экземпляром

[1, 4, 9, 16, 25]
[]
[1, 4, 9, 16, 25]


In [22]:
# __iter__ – генератор: каждый вызов iter() создаёт новый генератор
class SquaresRange:
    def __init__(self, start, stop):
        self.start, self.stop = start, stop

    def __iter__(self):
        for i in range(self.start, self.stop + 1):
            yield i ** 2

X = SquaresRange(1, 5)
print(list(X))   # первый проход
print(list(X))   # тот же объект – снова полный проход

gen = iter(X)    # сам генератор одноразовый, как Squares
print(list(gen), list(gen))

[1, 4, 9, 16, 25]
[1, 4, 9, 16, 25]
[1, 4, 9, 16, 25] []


### Перехват оператора `in`: `__contains__`, `__iter__`, и `__getitem__`

- В Python оператор `in` реализуется через **многоуровневую систему вызовов**:
  1. Если в классе определён `__contains__`, используется **он** (приоритетный способ).
  2. Если нет – вызывается `__iter__`, и выполняется поиск по итерации.
  3. Если нет и `__iter__` – используется `__getitem__`,  
     который перебирает элементы с индексами, пока не возникнет `IndexError`.

- Таким образом, класс может:
  - реализовать **оптимизированную проверку принадлежности** (`__contains__`);
  - или опираться на **итерацию** (`__iter__`);
  - или использовать **индексирование как резервный вариант** (`__getitem__`).

- В примере цепочку видно по шагам: методы по одному удаляются из класса,
  и `in` переходит к следующему способу.


In [30]:
# Демонстрация цепочки: __contains__, __iter__, __getitem__

def trace(msg):
    """Утилита для наглядного вывода шагов"""
    print(msg, end=" ")

class Iters:
    def __init__(self, value):
        self.data = value

    def __getitem__(self, i):
        trace(f"@get[{i}]")
        return self.data[i]

    def __iter__(self):
        trace("@iter")
        self.ix = 0
        return self

    def __next__(self):
        trace("@next")
        if self.ix == len(self.data):
            raise StopIteration
        item = self.data[self.ix]
        self.ix += 1
        return item

    def __contains__(self, x):
        trace("@contains")
        return x in self.data


X = Iters([1, 2, 3, 4])
print(3 in X)            # есть __contains__ – вызван он

del Iters.__contains__   # убираем методы по одному
print(3 in X)            # поиск идёт через __iter__ и __next__

del Iters.__iter__
print(3 in X)            # остался только __getitem__

@contains True
@iter @next @next @next True
@get[0] @get[1] @get[2] True


### Перехват доступа к атрибутам: `__getattr__`, `__setattr__`, `__delattr__`

- `obj.x` → если `x` **не найден** обычным поиском атрибутов (см. лекцию 7), вызывается `__getattr__(self, name)`.
- `obj.x = v` → всегда вызывает `__setattr__(self, name, value)` (перехватывает **все** присваивания).
- `del obj.x` → вызывает `__delattr__(self, name)`.
- Внутри `__setattr__` запись `self.x = ...` снова вызовет этот метод – та же рекурсия,
  что у сеттера, который пишет в своё же имя (лекция 7). Поэтому пишут
  `object.__setattr__(self, name, value)` или `self.__dict__[name] = value`.
- Если атрибута нет или доступ к нему запрещён, метод выбрасывает `AttributeError` (`raise`), а не возвращает объект ошибки.
- Что выбрать: проверка одного атрибута – `property` (лекция 7); правило для всех
  присваиваний – `__setattr__`; передача неизвестных атрибутов другому объекту –
  `__getattr__` (делегирование, ниже).

In [31]:
# Динамический атрибут: если атрибут не найден – вычисляем "на лету"
class Empty:
    def __getattr__(self, name):
        if name == "age":
            return 40                    # поля age в объекте нет, но X.age вернёт 40
        raise AttributeError(name)       # сигнал "нет такого атрибута"

X = Empty()
print("X.age ->", X.age)
try:
    print("X.name ->", X.name)
except AttributeError as e:
    print("X.name: ошибка –", e)


X.age -> 40
X.name: ошибка – name


### Возврат `None` скрывает опечатку

Если `__getattr__` возвращает `None` для любого неизвестного имени,
чтение атрибута считается успешным. Опечатка превращается в обычное значение.

Поэтому `hasattr` отвечает `True`, а `getattr` не использует запасное значение.
Такой же эффект даёт забытый `raise`: функция без `return` вернёт `None`.

In [1]:
class SilentAttrs:
    def __getattr__(self, name):
        return None


silent = SilentAttrs()
silent.age = 20
print("Опечатка agge:", silent.agge)
print("Атрибут существует:", hasattr(silent, "agge"))
print("С запасным значением:", getattr(silent, "agge", "не найден"))

Опечатка agge: None
Атрибут существует: True
С запасным значением: None


### Отсутствие атрибута: `hasattr` и значение по умолчанию

В уже написанном `Empty` неизвестное имя приводит к `raise AttributeError`.

- `hasattr(obj, name)` пытается прочитать атрибут: при `AttributeError` вернёт `False`.
- `getattr(obj, name, default)` при отсутствии атрибута вернёт `default`.
- Другие исключения эти функции не скрывают.

Существующее значение `None` не означает, что атрибут отсутствует.

In [2]:
person = Empty()
print("Есть age:", hasattr(person, "age"))
print("Есть agge:", hasattr(person, "agge"))
print("Нет имени:", getattr(person, "name", "не указано"))

person.nickname = None
print("nickname существует:", getattr(person, "nickname", "не указано"))

Есть age: True
Есть agge: False
Нет имени: не указано
nickname существует: None


In [32]:
# __setattr__: одно правило для всех присваиваний
class Settings:
    allowed = {"host", "port"}          # других полей у настроек быть не может

    def __setattr__(self, name, value):
        if name not in self.allowed:
            raise AttributeError(f"неизвестная настройка {name!r}")
        object.__setattr__(self, name, value)   # не self.name = value: снова вызвало бы __setattr__

s = Settings()
s.host = "localhost"
s.port = 8080
print(vars(s))
try:
    s.hots = "example.com"              # опечатка в имени
except AttributeError as error:
    print("AttributeError:", error)

{'host': 'localhost', 'port': 8080}
AttributeError: неизвестная настройка 'hots'


### Дескрипторы: свойство, которое можно переиспользовать

`property` из лекции 7 привязан к одному атрибуту: для второго поля с той же
проверкой пришлось бы писать второе свойство.

**Дескриптор** – объект отдельного класса с методами `__get__`/`__set__`, присвоенный
атрибуту класса: в примере `price = Positive()` в теле `Product`. Такой атрибут
перехватывает чтение и запись `item.price` у всех экземпляров `Product`, поэтому один
класс проверки подходит для любого числа полей. `__set_name__` Python вызывает при
создании класса и передаёт имя поля. Сам `property` устроен как дескриптор.

`__setattr__` и `__getattribute__` пишут в самом классе, и они перехватывают **все**
атрибуты его экземпляров. Дескриптор отвечает за **одно** поле и живёт в отдельном классе.

In [28]:
class Positive:
    """Дескриптор: значение поля должно быть положительным."""

    def __set_name__(self, owner, name):       # вызывается при создании класса
        self.name = name
        self.storage = "_" + name              # значение храним под другим именем

    # self – дескриптор (один на поле класса), instance – объект Product,
    # у которого читают или пишут поле, owner – сам класс Product
    def __get__(self, instance, owner):
        if instance is None:                   # обращение через класс: Product.price
            return self
        return getattr(instance, self.storage)

    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError(f"{self.name} должно быть положительным")
        setattr(instance, self.storage, value)


class Product:
    price = Positive()                         # одна проверка –
    weight = Positive()                        # на два поля

    def __init__(self, price, weight):
        self.price = price                     # запись идёт через Positive.__set__
        self.weight = weight


item = Product(250, 1.5)
print(item.price, item.weight)
try:
    item.weight = 0
except ValueError as error:
    print("ValueError:", error)
print(vars(item))

250 1.5
ValueError: weight должно быть положительным
{'_price': 250, '_weight': 1.5}


### Строковое представление: `__repr__` и `__str__`

- `print(obj)` / `str(obj)` → сначала пытаются `__str__`, при его отсутствии – `__repr__`.
- Все прочие контексты (REPL-эхо, `repr()`, вложенные объекты, контейнеры) → `__repr__`.
- Назначение:
  - `__str__` – «для пользователя» (кратко и понятно).
  - `__repr__` – «для разработчика» (детально, желательно «as-code»).
- Оба метода обязаны возвращать `str` (иначе ошибка).
- В контейнерах (списки, кортежи, словари) отображение элементов идёт через `__repr__`.

- Рекомендации:
  - Один универсальный формат – достаточно `__repr__`.
  - Два разных формата («пользовательский» и «разработческий») – следует реализовать оба.


In [36]:
# __str__ – для пользователя, __repr__ – для разработчика
class Price:
    def __init__(self, amount, currency):
        self.amount = amount
        self.currency = currency

    def __repr__(self):
        return f"Price({self.amount!r}, {self.currency!r})"

    def __str__(self):
        return f"{self.amount} {self.currency}"


price = Price(250, "руб.")
print(price)                 # print → __str__
print(repr(price))           # repr → __repr__
print([price, price])        # элементы списка показываются через __repr__
print(f"{price} и {price!r}")


class OnlyRepr:
    def __repr__(self):
        return "OnlyRepr()"

print(OnlyRepr())            # __str__ нет – print берёт __repr__

250 руб.
Price(250, 'руб.')
[Price(250, 'руб.'), Price(250, 'руб.')]
250 руб. и Price(250, 'руб.')
OnlyRepr()


### Правые и «на месте» операции: `__radd__` и `__iadd__`

- У бинарного оператора три метода: обычный (`__add__`), **правый** (`__radd__` –
  экземпляр стоит справа: `1 + x`) и **«на месте»** (`__iadd__` – для `+=`).
  Так же устроены `-`, `*` и другие: `__sub__`, `__rsub__`, `__isub__`…
- `a + b` сначала пробует `a.__add__(b)`; если тот не справился – `b.__radd__(a)`.
  Если тип справа – подкласс типа слева со своим `__radd__`, правый метод
  получает приоритет.
- `a += b` сначала ищет `__iadd__`, а без него выполняет `a = a + b`.
- Правые и «на месте» методы нужны там, где операции смешивают типы или выгодно
  менять объект на месте: векторы, матрицы, числовые контейнеры.

In [37]:
# Базовый пример: левая/правая сторона для +
class Commuter1:
    def __init__(self, val):
        self.val = val
    def __add__(self, other):
        print("add ", self.val, other)
        return self.val + other              # для простоты вернём число
    def __radd__(self, other):
        print("radd", self.val, other)
        return other + self.val
    def __repr__(self):
        return f"Commuter1({self.val})"

x = Commuter1(88)
y = Commuter1(99)
print("x + 1 =", x + 1)      # add
print("1 + y =", 1 + y)      # radd
# x + y: внутри x.__add__ выполняется 88 + y – число слева, объект справа,
# поэтому дальше Python вызывает y.__radd__(88)
print("x + y =", x + y)

add  88 1
x + 1 = 89
radd 99 1
1 + y = 100
add  88 Commuter1(99)
radd 99 88
x + y = 187


### `NotImplemented` передаёт операцию другому типу

`Score` складывает свои экземпляры. Для другого типа он возвращает
специальное значение `NotImplemented`; тогда правый операнд получает шанс
через `__radd__`. В примере `Score` и `Bonus` не связаны наследованием.

In [2]:
class Score:
    def __init__(self, value):
        self.value = value

    def __add__(self, other):
        if not isinstance(other, Score):
            print("Score.__add__: тип не поддержан")
            return NotImplemented
        return Score(self.value + other.value)

class Bonus:
    def __init__(self, value):
        self.value = value

    def __radd__(self, other):
        if not isinstance(other, Score):
            return NotImplemented
        print("Bonus.__radd__: операция поддержана")
        return Score(other.value + self.value)

score = Score(10)
print((score + Bonus(3)).value)

Score.__add__: тип не поддержан
Bonus.__radd__: операция поддержана
13


### `NotImplemented` и `NotImplementedError` – разные вещи

Первое **возвращают** из специального метода: Python продолжает поиск
реализации. Если сложение не поддержал ни один операнд, оператор выдаёт `TypeError`.
`NotImplementedError` – **исключение**: его выброс остановит операцию
без перехода к правому методу. Абстрактные методы с ним разбирали в лекции 7.

In [3]:
print("Прямой вызов вернул маркер:",
      Score.__add__(score, "10") is NotImplemented)
try:
    score + "10"
except TypeError:
    print("Оператор + не нашёл подходящей реализации")

try:
    raise NotImplemented
except TypeError:
    print("NotImplemented не является исключением")

Score.__add__: тип не поддержан
Прямой вызов вернул маркер: True
Score.__add__: тип не поддержан
Оператор + не нашёл подходящей реализации
NotImplemented не является исключением


In [38]:
# Переиспользование логики: alias radd = add 
class CommuterAlias:
    def __init__(self, val):
        self.val = val
    def __add__(self, other):
        return self.val + (other.val if isinstance(other, CommuterAlias) else other)
    __radd__ = __add__  # обе стороны одинаковы

ca1, ca2 = CommuterAlias(10), CommuterAlias(5)
print("ca1 + 7   =", ca1 + 7)
print("7 + ca1   =", 7 + ca1)            # работает за счёт alias
print("ca1 + ca2 =", ca1 + ca2)

ca1 + 7   = 17
7 + ca1   = 17
ca1 + ca2 = 15


### `+=`: изменить объект или создать новый

С `__iadd__` объект меняется на месте, и изменение видят все ссылки на него.
Без `__iadd__` Python выполняет `a = a + b`: создаётся новый объект, имя `a`
переходит на него, а другие ссылки остаются на старом.

Это та же разница, что у списка в лекции 4: `L += [3, 4]` меняет список,
который видит и `M`, а `L = L + [3, 4]` – нет. Список ведёт себя так потому,
что у него есть `__iadd__`.

In [34]:
class NumberIAdd:
    def __init__(self, val):
        self.val = val
    def __iadd__(self, other):
        self.val += other       # меняем объект на месте
        return self             # обычно возвращают self
    def __repr__(self):
        return f"NumberIAdd({self.val})"


class NumberAddFallback:        # __iadd__ нет: += выполнит nf = nf + 1
    def __init__(self, val):
        self.val = val
    def __add__(self, other):
        return NumberAddFallback(self.val + other)
    def __repr__(self):
        return f"NumberAddFallback({self.val})"


ni = NumberIAdd(5)
same_ni = ni
ni += 1
print("с __iadd__:   ", ni, "| вторая ссылка:", same_ni)

nf = NumberAddFallback(5)
same_nf = nf
nf += 1
print("без __iadd__: ", nf, "| вторая ссылка:", same_nf)

с __iadd__:    NumberIAdd(6) | вторая ссылка: NumberIAdd(6)
без __iadd__:  NumberAddFallback(6) | вторая ссылка: NumberAddFallback(5)


### Перегрузка вызова: `__call__` – «экземпляр как функция»

- Если в классе определить `__call__(self, *args, **kwargs)`, экземпляры можно **вызывать** как функции: `obj(...)`.
- Python передаёт в `__call__` все позиционные/именованные аргументы, поддерживаются *args/**kwargs.
- Зачем это нужно:
  - **Сохранение состояния** между вызовами: объект хранит параметры/кэш/счётчики.
    То же умеет замыкание (лекция 6), но у объекта состояние видно снаружи:
    `p2.calls` на следующем экране.
  - **Совместимость с API**, ожидающими «функцию» (колбэки, обработчики событий и данных).
  - **Обёртки** (декораторы-объекты): добавляют логику вокруг вызова другой функции.

In [43]:
# перехват вызова экземпляра
class Callee:
    def __call__(self, *pargs, **kargs):
        return {"args": pargs, "kwargs": kargs}

c = Callee()
print("Callee():", c(1, 2, x=3))

Callee(): {'args': (1, 2), 'kwargs': {'x': 3}}


In [44]:
# «умножитель», помнит коэффициент
class Prod:
    def __init__(self, factor):
        self.factor = factor
        self.calls = 0
    def __call__(self, value):
        self.calls += 1
        return self.factor * value

p2 = Prod(2)
print("2 * 3 =", p2(3), "| calls:", p2.calls)
print("2 * 4 =", p2(4), "| calls:", p2.calls)

2 * 3 = 6 | calls: 1
2 * 4 = 8 | calls: 2


In [46]:
# Объект-декоратор: оборачивает функцию и добавляет поведение
class Logger:
    def __init__(self, func, tag="LOG"):
        self.func = func
        self.tag = tag
        self.count = 0
    def __call__(self, *args, **kwargs):
        self.count += 1
        result = self.func(*args, **kwargs)
        call = f"{self.func.__name__}({', '.join(map(repr, args))})"
        print(f"[{self.tag}] вызов №{self.count}: {call} -> {result}")
        return result

def square(x): 
    return x * x

logged_square = Logger(square, tag="SQR")
print("logged_square(4) =", logged_square(4))
print("logged_square(5) =", logged_square(5))

[SQR] вызов №1: square(4) -> 16
logged_square(4) = 16
[SQR] вызов №2: square(5) -> 25
logged_square(5) = 25


### Логические проверки: `__bool__` и `__len__`

- Любой объект в Python имеет булево значение – при `if`, `while`, `bool(obj)` и т.д.  
- Кастомные классы могут **определять собственную «истинность»** с помощью:
  - `__bool__(self)` → возвращает `True` или `False`;
  - `__len__(self)` → используется как **запасной вариант**, если `__bool__` отсутствует  
    (ноль – `False`, не ноль – `True`).
- Приоритет:  
  `__bool__` → `__len__` → по умолчанию `True`, если оба отсутствуют.
- Использование:
  - Объекты с внутренним состоянием («пустой»/«валидный») могут сами определять, считаются ли они истинными.
  - Делает код естественным: `if basket:` вместо `if len(basket) > 0:`.
- `__bool__` обязан вернуть именно `bool`: `int` вызовет `TypeError`.


In [47]:
# Явная логика истинности через __bool__
class Truth:
    def __bool__(self):
        print("__bool__ called")
        return True

t = Truth()
if t:
    print("yes!")  # т.к. __bool__ → True


__bool__ called
yes!


In [49]:
# Запасной вариант: __len__
class MaybeEmpty:
    def __init__(self, data):
        self.data = list(data)
    def __len__(self):
        print("__len__ called")
        return len(self.data)

a = MaybeEmpty([])
b = MaybeEmpty([1, 2, 3])
if not a:
    print("a – пуст, значит False")
if b:
    print("b – непуст, значит True")

__len__ called
a – пуст, значит False
__len__ called
b – непуст, значит True


In [50]:
# __bool__ имеет преимущество над __len__
class Dual:
    def __bool__(self):
        print("__bool__ called")
        return True
    def __len__(self):
        print("__len__ called")
        return 0

d = Dual()
if d:
    print("использован __bool__, а не __len__")

__bool__ called
использован __bool__, а не __len__


### Уничтожение объектов: `__del__`

`__del__` – **финализатор**: Python может вызвать его перед уничтожением объекта.

- `del name` удаляет имя, а не вызывает `__del__` напрямую.
- Пока объект достижим, сборщик мусора его не уничтожает.
- Недостижимый цикл ссылок может быть собран позже; момент финализации
  не стоит использовать как часть логики программы.
- Исключение из `__del__` не передаётся вызывающему коду: о нём выводится
  сообщение в `stderr`.
- При завершении интерпретатора вызов `__del__` не гарантирован.

Файлы и соединения своевременно закрывает `with` (контекстные менеджеры
разобраны выше) или явный `close()` с `try/finally`.


In [52]:
# Пример деструктора
class Life:
    def __init__(self, name):
        print("Создан", name)
        self.name = name
    def live(self):
        print(self.name, "работает")
    def __del__(self):
        print("Удалён", self.name)

robot = Life("Робик")
robot.live()

# Убираем последнюю ссылку; в CPython здесь обычно сразу вызывается __del__
robot = "end"

Создан Робик
Робик работает
Удалён Робик


In [2]:
# Непредсказуемость вызова __del__
import gc       # модуль сборщика мусора: gc.collect() ищет циклы ссылок

class CycleNode:
    def __init__(self, name, events):
        self.name = name
        self.events = events

    def __del__(self):
        self.events.append(self.name)


collected = []
a = CycleNode("A", collected)
b = CycleNode("B", collected)
a.link = b
b.link = a
print("Цикл создан; имена a и b сохраняют доступ к объектам")


Цикл создан; имена a и b сохраняют доступ к объектам


In [3]:
gc.collect()  # Принудительный сбор мусора
print("Пока существуют a и b:", sorted(collected))   # объекты достижимы через имена

# Обычно объект освобождается, как только на него не остаётся ссылок. После del
# узлы всё ещё ссылаются друг на друга, поэтому счётчики ссылок не обнуляются –
# такой цикл находит только сборщик gc
del a, b
gc.collect()
print("После удаления внешних ссылок:", sorted(collected))

Пока существуют a и b: []
После удаления внешних ссылок: ['A', 'B']


## Проектирование с использованием классов

Когда один класс использует другой, нужно **выбрать отношение** между ними:

- «является» (is-a) – наследование;
- «содержит» (has-a) – композиция;
- «ведёт себя как он, но с добавкой» (like-a) – делегирование через обёртку.

### Наследование и отношения «is-a»

Наследование выбираем, если фраза **«X – это Y»** звучит естественно
и объект X можно использовать везде, где ожидается Y.

Повар – это сотрудник, поэтому `Chef` может наследовать `Employee`.
Если X не является Y, а только содержит его, нужна композиция.

### Композиция и отношения «has-a»

Композицию выбираем, если фраза **«X содержит Y»** описывает модель точнее,
чем «X – это Y». Пиццерия не является официантом, поваром или печью –
она содержит эти объекты и управляет порядком их работы.

Наследование и композиция могут использоваться вместе: это разные отношения.

In [58]:
# Пиццерия как составной объект (композиция)

class Server:
    def __init__(self, name):
        self.name = name


class Cook:
    def __init__(self, name):
        self.name = name

    def work(self):
        print(self.name, "готовит пиццу")


class Customer:
    def __init__(self, name):
        self.name = name

    def order(self, server):
        print(self.name, "делает заказ у", server.name)

    def pay(self, server):
        print(self.name, "оплачивает заказ у", server.name)


class Oven:
    def bake(self):
        print("печь выпекает пиццу")


class PizzaShop:
    def __init__(self):
        # Вложенные объекты
        self.server = Server("Ян")
        self.cook = Cook("Робик")
        self.oven = Oven()

    def order(self, name):
        customer = Customer(name)
        customer.order(self.server)
        self.cook.work()
        self.oven.bake()
        customer.pay(self.server)


scene = PizzaShop()
scene.order("Света")
print("...")
scene.order("Борис")

Света делает заказ у Ян
Робик готовит пиццу
печь выпекает пиццу
Света оплачивает заказ у Ян
...
Борис делает заказ у Ян
Робик готовит пиццу
печь выпекает пиццу
Борис оплачивает заказ у Ян


### Делегирование и отношения «like-a»

- **Делегирование** – это форма композиции, при которой объект передаёт выполнение действий другому объекту.  
- Используется, когда нужно **контролировать или расширять** поведение существующего объекта.  
- Контроллер (или обёртка) хранит внутри себя другой объект и **передаёт ему вызовы методов**.  
- В Python это удобно реализуется через метод `__getattr__()`, который перехватывает обращения к отсутствующим атрибутам.  
- Делегирование позволяет добавлять дополнительные шаги – например, **логирование**, **валидацию** или **мониторинг**.  
- Такое отношение условно называют **«like-a»** («похож на»): объект ведёт себя **как другой**, но с дополнительной логикой.  


In [59]:
# Делегирование через обёртку

class Wrapper:
    def __init__(self, obj):
        self.wrapped = obj  # Сохраняем вложенный объект

    def __getattr__(self, attrname):
        print("перехвачено:", attrname)          # сообщение при обращении к атрибуту
        return getattr(self.wrapped, attrname)  # Делегируем доступ к вложенному объекту


print("=== Работа со списком ===")
x = Wrapper([1, 2, 3])
x.append(4)         # Вызывает list.append
print(x.wrapped)

print("\n=== Работа со словарём ===")
y = Wrapper({'a': 1, 'b': 2})
print(list(y.keys()))  # Вызывает dict.keys

=== Работа со списком ===
перехвачено: append
[1, 2, 3, 4]

=== Работа со словарём ===
перехвачено: keys
['a', 'b']


### Обычный доступ работает, `len()` – нет

Наша обёртка `Wrapper` передаёт обычное чтение атрибутов вложенному объекту.
Даже явный вызов `wrapped_list.__len__()` пройдёт через `__getattr__`.

Но `len(wrapped_list)` ищет специальный метод в **классе обёртки**
и его родителях, обходя такой перехват. Автоматического переноса
всех операций вложенного объекта не происходит.

In [3]:
wrapped_list = Wrapper([10, 20])
print("Явный вызов:", wrapped_list.__len__())

try:
    print(len(wrapped_list))
except TypeError:
    print("У класса Wrapper нет __len__")

перехвачено: __len__
Явный вызов: 2
У класса Wrapper нет __len__


### Явно делегируем длину обёртки

Определим `__len__` в подклассе: он запросит длину вложенного объекта.
Обычные методы по-прежнему передаёт унаследованный `__getattr__`.

После изменения списка длина должна отражать его текущее состояние.
Остальные операции, например индексацию, такая обёртка сама по себе не получает.

In [4]:
class SizedWrapper(Wrapper):
    def __len__(self):
        return len(self.wrapped)


sized_list = SizedWrapper([10, 20])
print("До добавления:", len(sized_list))
sized_list.append(30)
print("После добавления:", len(sized_list))

До добавления: 2
перехвачено: append
После добавления: 3


### Псевдоприватные атрибуты класса

- В Python **все атрибуты по умолчанию открыты (public)** – их можно читать и изменять извне.  
- Все атрибуты экземпляра лежат в **одном общем пространстве**, поэтому при наследовании
  разные классы легко занимают одно и то же имя и перезаписывают данные друг друга:
  `self.attr = ...` всегда пишет в одно и то же поле.  
- Чтобы этого избежать, есть механизм **псевдоприватных имён** – *name mangling*.  
- Имена, начинающиеся с двух подчёркиваний (`__name`), внутри класса автоматически
  преобразуются: `__attr` → `_ClassName__attr`.  
- Это **не защита**, а способ локализовать имя внутри класса.  
- В обычном коде чаще используют **одно подчёркивание** (`_name`) – соглашение
  «внутреннее, руками не трогать», разбиралось в лекции про пространства имён.


In [60]:
# Пример работы механизма "name mangling"

class Base:
    def __init__(self):
        self.__hidden = "секрет базового класса"

    def reveal(self):
        print("Base:", self.__hidden)


class Sub(Base):
    def __init__(self):
        super().__init__()
        self.__hidden = "секрет подкласса"


obj = Sub()

# reveal написан в Base: там self.__hidden – это _Base__hidden.
# Sub записал своё значение в _Sub__hidden и поле базового класса не затёр
obj.reveal()


Base: секрет базового класса


In [61]:
# Прямой доступ к атрибутам
print(obj._Base__hidden)   # Псевдоприватное имя из Base
print(obj._Sub__hidden)    # Псевдоприватное имя из Sub

секрет базового класса
секрет подкласса


In [62]:
# Попытка обратиться напрямую без преобразования 
try:
    print(obj.__hidden)
except AttributeError as e:
    print("\nОшибка при доступе к __hidden напрямую:", e)


Ошибка при доступе к __hidden напрямую: 'Sub' object has no attribute '__hidden'


### Объекты-методы: привязанные и непривязанные

Как метод получает `self` при вызове через объект и через класс, разбирали в лекции 7
(экран «Вызов методов»). Здесь – другая сторона того же механизма:

- **Методы** в Python – объекты, как функции или строки: их можно сохранять,
  передавать, вызывать.
- `obj.method` без скобок – **привязанный метод (bound method)**: отдельный объект,
  который запомнил экземпляр и при вызове сам передаст его в `self`.
- `Class.method` – **обычная функция (plain function)**.
- Привязанный метод можно сохранить в переменную, положить в список и вызвать позже –
  он по-прежнему помнит свой объект. На этом строятся колбэки и таблицы действий,
  их примеры – на следующих экранах.

In [65]:
# Разница между привязанными и непривязанными методами

class Demo:
    def greet(self, msg):
        print(f"{self.name} говорит: {msg}")

# Создаём экземпляр
obj = Demo()
obj.name = "Питон"

# Доступ к методу через экземпляр – создаётся "bound method"
bound = obj.greet
print("Привязанный метод:", bound)
bound("Привет!")  # self передаётся автоматически

Привязанный метод: <bound method Demo.greet of <__main__.Demo object at 0x10cc37b60>>
Питон говорит: Привет!


In [66]:
# Доступ к тому же методу через класс – "plain function"
plain = Demo.greet
print("\nФункция из класса:", plain)
plain(obj, "Здравствуйте!")  # self нужно передать вручную


Функция из класса: <function Demo.greet at 0x10cd0ef20>
Питон говорит: Здравствуйте!


In [69]:
# Хранение bound методов в коллекции

class Number:
    def __init__(self, base):
        self.base = base

    def double(self):
        return self.base * 2

    def triple(self):
        return self.base * 3


x, y, z = Number(2), Number(3), Number(4)
acts = [x.double, y.double, z.double, z.triple]  # список привязанных методов

print("\nВызовы методов из списка:")
for act in acts:
    print(act(), end=' ')



Вызовы методов из списка:
4 6 8 12 

### Привязанные методы как колбэки

Привязанный метод можно передать туда, где ждут функцию (экраны выше). Типичный
пример – **обработчики событий (callbacks)** в графических интерфейсах: кнопке
`tkinter` в параметр `command=` передают функцию без аргументов, которая вызывается
при нажатии. `command=self.handler` – привязанный метод, поэтому при вызове у него
есть `self`, и счётчик нажатий `self.clicks` живёт в объекте.

In [70]:
# Использование bound метода как callback в tkinter

import tkinter as tk

class MyGui:
    def __init__(self):
        self.clicks = 0
        self.make_widgets()

    def handler(self):
        """Метод-обработчик: вызывается при нажатии кнопки"""
        self.clicks += 1
        print(f"Кнопка нажата {self.clicks} раз(а)")

    def make_widgets(self):
        """Создание интерфейса"""
        root = tk.Tk()
        root.title("Привязанный метод как обработчик")

        btn = tk.Button(root, text="Нажми меня", command=self.handler)
        btn.pack(padx=20, pady=20)

        root.mainloop()


# Окно открывается вызовом MyGui(); в ноутбуке ячейка ждала бы, пока его не закроют.
# Чтобы увидеть окно, сохраните код в файл .py, добавьте в конце MyGui() и запустите

## Классы-данные: `dataclasses`

Часто класс нужен просто как **контейнер данных** – с `__init__`, `__repr__`, `__eq__`,
которые пишутся однотипно и скучно. Декоратор `@dataclass` **генерирует их автоматически**
по аннотациям полей.

In [74]:
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int = 0          # значение по умолчанию

p = Point(1, 2)
print(p)                # авто __repr__: Point(x=1, y=2)
print(p == Point(1, 2)) # авто __eq__: True
print(Point(5))         # y берётся по умолчанию: Point(x=5, y=0)

Point(x=1, y=2)
True
Point(x=5, y=0)


### Что генерирует `@dataclass`

Упрощённый вариант вручную для этих полей:

```python
class Point:
    def __init__(self, x, y=0):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"Point(x={self.x!r}, y={self.y!r})"
    def __eq__(self, other):
        if type(other) is not type(self):
            return NotImplemented
        return self.x == other.x and self.y == other.y
```

`@dataclass` умеет и больше: `order=True` добавит все сравнения
(`<`, `<=`, `>`, `>=`) по кортежу полей – как `total_ordering` в разделе сравнений;
`frozen=True` запретит присваивание и удаление полей обычным способом.
Вложенные объекты он не замораживает.

### Аннотации полей не проверяют входные данные

У уже определённого `Point` поле `x` аннотировано как `int`.
Но сгенерированный `__init__` просто сохраняет переданный объект:
`dataclass` сам не проверяет тип и не преобразует строку в число.

In [4]:
raw_point = Point("2", 3)
print("Тип x:", type(raw_point.x).__name__)
print("Значение x:", repr(raw_point.x))

Тип x: str
Значение x: '2'


### Проверка после создания полей: `__post_init__`

`__post_init__` – не общий специальный метод Python, а договорённость модуля
`dataclasses`: сгенерированный `__init__` присваивает поля и, если в классе есть
`__post_init__`, вызывает его. Здесь проверяют типы и значения. Нет метода – нет
и проверки: поля просто сохраняются, как на предыдущем экране.

Почему не в `__init__`: его пишет за нас `dataclass`. Свой `__init__` он не заменит,
но тогда поля придётся присваивать самим, а `__post_init__` вызываться перестанет.

In [5]:
@dataclass
class Session:
    seats: int

    def __post_init__(self):
        if type(self.seats) is not int:
            raise TypeError("Число мест должно быть целым")
        if self.seats <= 0:
            raise ValueError("Число мест должно быть положительным")

print(Session(3))
for bad in ("3", 0, True):
    try:
        Session(bad)
    except (TypeError, ValueError) as error:
        print(type(error).__name__, str(error))

Session(seats=3)
TypeError Число мест должно быть целым
ValueError Число мест должно быть положительным
TypeError Число мест должно быть целым


### Изменяемые поля и `frozen`

- Изменяемое значение по умолчанию (`list`, `dict`) `dataclass` не примет: `items: list = []`
  даёт `ValueError` ещё при определении класса. Это защита от общего списка на все
  экземпляры (ловушка из лекций 5 и 7). Правильно – `field(default_factory=list)`.
- `frozen=True` запрещает присваивание и удаление полей (`FrozenInstanceError`),
  но не изменение вложенных объектов. При стандартном `eq=True` dataclass создаёт
  `__hash__`; его вычисление требует хешируемости всех участвующих полей.

In [75]:
from dataclasses import dataclass, field

@dataclass
class Basket:
    items: list = field(default_factory=list)   # у каждого экземпляра свой список
    def add(self, x):
        self.items.append(x)

b1, b2 = Basket(), Basket()
b1.add("яблоко")
print("b1:", b1.items, "| b2:", b2.items)

@dataclass(frozen=True)
class Config:
    host: str
    port: int = 8000

c = Config("localhost")
print(c)
try:
    c.port = 9000                 # frozen → изменить нельзя
except Exception as e:
    print(type(e).__name__, "-", e)

b1: ['яблоко'] | b2: []
Config(host='localhost', port=8000)
FrozenInstanceError - cannot assign to field 'port'


### `frozen` не замораживает вложенные объекты

У `frozen`-экземпляра нельзя переприсвоить поле, но список в этом поле
по-прежнему изменяем. Автоматический `__hash__` также не сделает список
хешируемым: попытка вычислить хеш всего объекта завершится ошибкой.

In [6]:
@dataclass(frozen=True)
class ReadingList:
    titles: list[str] = field(default_factory=list)

reading = ReadingList(["Python"])
reading.titles.append("Алгоритмы")
print(reading.titles)
try:
    reading.titles = []
except AttributeError:
    print("Переприсвоить поле нельзя")
try:
    hash(reading)
except TypeError:
    print("Поле-список не позволяет вычислить хеш")

['Python', 'Алгоритмы']
Переприсвоить поле нельзя
Поле-список не позволяет вычислить хеш


### Ключ из неизменяемых хешируемых полей

Для набора названий подойдёт кортеж строк. В этом примере равенство и хеш
сгенерированы по одним и тем же полям; равный экземпляр находит запись в словаре.
Кортеж с вложенным списком такой гарантии не дал бы.

In [7]:
@dataclass(frozen=True)
class ReadingKey:
    titles: tuple[str, ...]

key = ReadingKey(("Python", "Алгоритмы"))
equal_key = ReadingKey(tuple(["Python", "Алгоритмы"]))
print("Равны:", key == equal_key)
print("Хеши равны:", hash(key) == hash(equal_key))
labels = {key: "основной список"}
print(labels[equal_key])

Равны: True
Хеши равны: True
основной список


### Служебное поле: `compare=False`

Одинаковые статьи могут быть загружены в разное время.
Если время загрузки не определяет равенство статей, исключим его
из сгенерированного сравнения через `field(compare=False)`.

Поле остаётся в объекте и конструкторе. Какие поля определяют равенство,
выбираем по смыслу задачи.

In [5]:
from dataclasses import dataclass, field


@dataclass
class Article:
    title: str
    text: str
    loaded_at: str = field(compare=False)


first = Article("Python", "Объекты и классы", "09:00")
second = Article("Python", "Объекты и классы", "09:05")
edited = Article("Python", "Новая редакция", "09:00")
print("Другое время:", first == second)
print("Другой текст:", first == edited)
print("Время сохранено:", second.loaded_at)

Другое время: True
Другой текст: False
Время сохранено: 09:05


### Короткое представление: `repr=False`

Большой текст может мешать читать вывод объекта.
`field(repr=False)` исключает поле из сгенерированного `__repr__`.

Поле по-прежнему доступно и участвует в сравнении, если не задан `compare=False`.
Это настройка отображения, а не ограничение доступа к данным.

In [6]:
@dataclass
class Document:
    title: str
    text: str = field(repr=False)


draft = Document("Отчёт", "Первая версия текста")
revised = Document("Отчёт", "Исправленная версия текста")
print(draft)
print("Текст доступен:", draft.text)
print("Представления совпали:", repr(draft) == repr(revised))
print("Документы равны:", draft == revised)

Document(title='Отчёт')
Текст доступен: Первая версия текста
Представления совпали: True
Документы равны: False


### Когда использовать `dataclass`

**Подходит**, если суть класса – **набор полей**: точки, настройки, записи, объекты
для передачи данных, узлы дерева – и нужны готовые `__init__`/`__repr__`/`__eq__`,
а иногда сравнение и неизменяемость.

**Не подходит**, если у класса сложная логика и мало данных – обычный класс уместнее.

Родственные инструменты: `typing.NamedTuple` (неизменяемые кортежи-записи),
`pydantic.BaseModel` (валидация данных – увидим в лекции про веб-разработку).


## Перечисления: `enum`

Когда у величины **фиксированный набор значений** (цвета, статусы, роли), вместо «магических»
строк/чисел используют `Enum`. Это даёт осмысленные имена, защиту от опечаток и удобное
сравнение по идентичности.

In [76]:
from enum import Enum

class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3

print(Color.RED)                            # Color.RED
print(Color.RED.name, "=", Color.RED.value) # RED = 1
print(Color(2))                             # поиск по значению: Color.GREEN
print("Все:", list(Color))

status = Color.RED
print(status is Color.RED)                  # сравнение по идентичности: True

Color.RED
RED = 1
Color.GREEN
Все: [<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 3>]
True


In [77]:
from enum import Enum, auto

class Status(Enum):
    ACTIVE = auto()      # значения назначаются автоматически: 1, 2, 3
    BLOCKED = auto()
    DELETED = auto()

for s in Status:
    print(s.name, s.value)

ACTIVE 1
BLOCKED 2
DELETED 3


### Из строки в `Enum`: проверяем внешний статус

Пусть входной словарь содержит строковый статус заказа.
Преобразуем его в `OrderStatus` при чтении данных и дальше работаем
с элементом перечисления.

Неизвестное значение вызывает `ValueError`: его нужно обработать,
а не молча заменять допустимым статусом. Для внешних кодов зададим
явные строковые значения.

In [7]:
from enum import Enum


class OrderStatus(Enum):
    NEW = "new"
    PAID = "paid"
    SENT = "sent"


incoming = {"status": "paid"}
order_status = OrderStatus(incoming["status"])
print("Принят статус:", order_status)

try:
    OrderStatus("paidd")
except ValueError:
    print("Неизвестный статус: paidd")

Принят статус: OrderStatus.PAID
Неизвестный статус: paidd


### Имя, значение и обратное преобразование `Enum`

`OrderStatus["PAID"]` ищет по **имени**, а `OrderStatus("paid")` –
по **значению**. Неизвестное имя вызывает `KeyError`; регистр имеет значение.

При подготовке выходных данных вернём строковый код через `.value`.
Имя члена перечисления и его внешний код могут различаться.

In [8]:
print("По имени:", OrderStatus["PAID"] is order_status)

try:
    OrderStatus["paid"]
except KeyError:
    print("paid – значение, а имя – PAID")

outgoing = {"status": order_status.value}
print("Для передачи:", outgoing)
print("Обратное чтение:", OrderStatus(outgoing["status"]) is order_status)

По имени: True
paid – значение, а имя – PAID
Для передачи: {'status': 'paid'}
Обратное чтение: True


## Частые ошибки

**`__post_init__` не проверяет последующие присваивания.** Проверка при создании
не делает изменяемый объект защищённым от любых неверных значений.
После создания значения защищают методы или свойства с проверками (лекция 7)
либо `frozen`, если по смыслу это неизменяемое значение.


In [8]:
session = Session(3)
session.seats = -1
print("После прямой записи:", session.seats)
print("__post_init__ автоматически не повторялся")


После прямой записи: -1
__post_init__ автоматически не повторялся


### `__eq__` без проверки типа

`Version.__eq__` сразу читает `other.parts`. Сравнение с объектом другого типа
падает с `AttributeError`, хотя должно дать `False`.

Для чужого типа метод возвращает `NotImplemented` (раздел про `__radd__`):
Python попробует метод другого операнда, а если и тот не поддержит сравнение,
`==` даст `False`.

In [74]:
try:
    Version(3, 12) == "3.12"
except AttributeError as error:
    print("AttributeError:", error)


class SafeVersion(Version):
    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts == other.parts


print(SafeVersion(3, 12) == "3.12")
print(SafeVersion(3, 12) == Version(3, 12))

AttributeError: 'str' object has no attribute 'parts'
False
True


### Элемент `Enum` не равен своему значению

`OrderStatus.PAID == "paid"` даёт `False` без ошибки – и условие молча
не срабатывает. Сравнивайте с элементом перечисления или переводите строку
в `Enum` при чтении данных, как в разделе про перечисления.

Если элементы должны совпадать со строками, есть `StrEnum` (Python 3.11+):
он наследует `str`.

In [75]:
from enum import StrEnum

print(OrderStatus.PAID == "paid")         # False: элемент Enum не равен строке
print(order_status is OrderStatus.PAID)   # сравниваем с элементом перечисления


class WebStatus(StrEnum):
    PAID = "paid"

print(WebStatus.PAID == "paid")           # StrEnum – подкласс str

False
True
True


## Мини-упражнения


### 1. `Temperature`: все сравнения

Класс `Temperature(celsius)` с `__repr__`. Сделайте так, чтобы работали **все**
сравнения (`==`, `<`, `<=`, `>`, `>=`) и сортировка. Сравнение с числом должно
давать `False` для `==` и `TypeError` для `<` – а не `AttributeError`.

In [76]:
# Решение: все сравнения через total_ordering и NotImplemented
import functools


@functools.total_ordering
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    def __repr__(self):
        return f"Temperature({self.celsius}°C)"

    def __eq__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.celsius == other.celsius

    def __lt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.celsius < other.celsius


print(sorted([Temperature(20), Temperature(-5), Temperature(10)]))
print(Temperature(10) <= Temperature(10), Temperature(20) > Temperature(-5))
print(Temperature(10) == 10)
try:
    Temperature(10) < 10
except TypeError:
    print("Temperature и число не сравниваются")

[Temperature(-5°C), Temperature(10°C), Temperature(20°C)]
True True
False
Temperature и число не сравниваются


### 2. Временное значение атрибута

Напишите через `@contextlib.contextmanager` функцию `temporary(obj, name, value)`:
внутри блока `with` атрибут `name` у `obj` равен `value`, а после блока
возвращается прежнее значение – **даже если в блоке возникло исключение**.

In [77]:
# Решение: временное значение через contextmanager и try/finally
import contextlib


@contextlib.contextmanager
def temporary(obj, name, value):
    old = getattr(obj, name)
    setattr(obj, name, value)
    try:
        yield obj
    finally:
        setattr(obj, name, old)          # восстанавливаем и при ошибке


class AppConfig:
    def __init__(self):
        self.debug = False


config = AppConfig()
with temporary(config, "debug", True):
    print("В блоке:", config.debug)
print("После блока:", config.debug)

try:
    with temporary(config, "debug", True):
        raise RuntimeError("сбой в блоке")
except RuntimeError:
    print("После ошибки:", config.debug)

В блоке: True
После блока: False
После ошибки: False


### 3. Длительность как значение и ключ

Создайте `@dataclass(frozen=True)` с именем `Duration` и полем `seconds: int`.

- В `__post_init__` отклоняйте всё, кроме неотрицательных целых чисел;
  `bool` тоже отклоняйте. Для неверных данных используйте `ValueError`.
- `__add__` складывает две длительности и возвращает новый `Duration`.
  Для чужого типа возвращайте `NotImplemented`.
- Проверьте сумму, ошибочный ввод и поиск в словаре по равному экземпляру.
  Собственный `__hash__` здесь писать не нужно.

Например, `Duration(2) + Duration(3)` равен `Duration(5)`, а сложение с `1`
не поддерживается. Исходные длительности должны сохранить значения.

### Решение: проверка, сложение и хеширование длительности

`dataclass` создаёт согласованные равенство и хеш. Наши методы отвечают
за допустимое значение и поддержанные операнды.

In [9]:
import pytest

@dataclass(frozen=True)
class Duration:
    seconds: int

    def __post_init__(self):
        if type(self.seconds) is not int or self.seconds < 0:
            raise ValueError("Нужны неотрицательные целые секунды")

    def __add__(self, other):
        if not isinstance(other, Duration):
            return NotImplemented
        return Duration(self.seconds + other.seconds)

left, right = Duration(2), Duration(3)
total = left + right
assert total == Duration(5)
assert (left.seconds, right.seconds) == (2, 3)
assert {total: "готово"}[Duration(5)] == "готово"
with pytest.raises(ValueError):
    Duration(-1)
with pytest.raises(TypeError):
    total + 1
print(total)

Duration(seconds=5)


## Справочник: специальные методы

Сводка методов этой лекции.

| Метод | Вызывается при |
|:------|:---------------|
| `__new__`, `__init__` | `Class(...)`: создание, затем заполнение объекта |
| `__repr__`, `__str__` | `repr(x)`; `print(x)`, `str(x)`, f-строки |
| `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__` | `==`, `!=`, `<`, `<=`, `>`, `>=` |
| `__hash__` | `hash(x)`, ключ словаря, элемент множества |
| `__bool__`, `__len__` | `bool(x)`, `if x:`; `len(x)` |
| `__add__`, `__sub__`, `__mul__`, `__or__`… | <code>x + y</code>, <code>x - y</code>, <code>x * y</code>, <code>x &#124; y</code> |
| `__radd__`, `__rsub__`… | `y + x`, если `y` не поддержал операцию |
| `__iadd__`, `__isub__`… | `x += y`, `x -= y` |
| `__getitem__`, `__setitem__`, `__delitem__` | `x[i]`, `x[i:j]`, `x[i] = v`, `del x[i]` |
| `__index__` | `L[x]`, `bin(x)`, `hex(x)`, `range(x)` |
| `__iter__`, `__next__` | `for`, `list(x)`, `iter(x)`, `next(x)` |
| `__contains__` | `y in x` |
| `__getattr__` | `x.name`, если обычный поиск атрибут не нашёл |
| `__getattribute__` | любое чтение `x.name` |
| `__setattr__`, `__delattr__` | `x.name = v`, `del x.name` |
| `__get__`, `__set__`, `__delete__`, `__set_name__` | дескриптор в атрибуте класса |
| `__call__` | `x(...)` |
| `__enter__`, `__exit__` | `with x as y:` |
| `__del__` | перед уничтожением объекта (без гарантий) |

## Что почитать

- [Модель данных](https://docs.python.org/3/reference/datamodel.html) – полный список
  «магических» методов; главная страница языка для тех, кто пишет свои типы.
- [`dataclasses`](https://docs.python.org/3/library/dataclasses.html) и
  [`enum`](https://docs.python.org/3/library/enum.html).
- Л. Рамальо, «Python. К вершинам мастерства» – перегрузка операторов и питонические объекты.
